# Chapter 5: Classification

**Author:** Ruben Alfa Richie
**Date:** September 29, 2026  
**Source:** Practical Statistics for Data Scientists (O'Reilly)

---

## Overview

Classification predicts categorical outcomes:
- Naive Bayes classifier
- Logistic regression
- Discriminant analysis
- Evaluating classification models (confusion matrix, ROC, AUC)
- Handling imbalanced data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, roc_auc_score

np.random.seed(42)
sns.set_style('whitegrid')

## 5.1 Logistic Regression

### Theory

**Logistic regression** predicts probability of binary outcome.

**Logistic function (sigmoid):**
$$p(y=1|x) = \frac{1}{1 + e^{-(b_0 + b_1x_1 + ... + b_px_p)}}$$

**Logit (log-odds):**
$$\log\left(\frac{p}{1-p}\right) = b_0 + b_1x_1 + ... + b_px_p$$

**Key Points:**
- Output: probability between 0 and 1
- Decision threshold (typically 0.5)
- Coefficients affect log-odds
- Uses maximum likelihood estimation

**Interpretation:**
- exp(b_j) = odds ratio for unit increase in x_j

In [ ]:
# Logistic regression example: loan default prediction
n = 500
np.random.seed(42)

# Features
credit_score = np.random.normal(650, 100, n)
income = np.random.normal(50000, 20000, n)
debt_ratio = np.random.uniform(0, 0.8, n)

# Generate default (1) or not (0) based on features
logit = -5 + 0.005*credit_score + 0.00002*income - 8*debt_ratio
prob = 1 / (1 + np.exp(-logit))
default = (np.random.rand(n) < prob).astype(int)

df = pd.DataFrame({
    'credit_score': credit_score,
    'income': income,
    'debt_ratio': debt_ratio,
    'default': default
})

print(f'Default rate: {default.mean():.1%}')
print(f'\nSample data:')
print(df.head())

# Train/test split
X = df[['credit_score', 'income', 'debt_ratio']]
y = df['default']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Fit logistic regression
model = LogisticRegression()
model.fit(X_train, y_train)

print(f'\nLogistic Regression Coefficients:')
for i, col in enumerate(X.columns):
    print(f'{col}: {model.coef_[0][i]:.6f} (OR: {np.exp(model.coef_[0][i]):.4f})')
print(f'Intercept: {model.intercept_[0]:.4f}')

# Predictions
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

print(f'\nTest Accuracy: {model.score(X_test, y_test):.3f}')

# Visualize sigmoid curve
plt.figure(figsize=(12, 5))
x_range = np.linspace(-10, 10, 100)
sigmoid = 1 / (1 + np.exp(-x_range))
plt.plot(x_range, sigmoid, linewidth=3)
plt.axhline(0.5, color='red', linestyle='--', label='Decision threshold')
plt.axvline(0, color='gray', linestyle='--', alpha=0.5)
plt.xlabel('Logit (log-odds)')
plt.ylabel('Probability')
plt.title('Logistic Function (Sigmoid)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 5.2 Confusion Matrix and Metrics

### Theory

**Confusion Matrix:**
```
                 Predicted
              Neg      Pos
Actual  Neg   TN       FP
        Pos   FN       TP
```

**Metrics:**
- **Accuracy** = (TP+TN) / Total
- **Precision** = TP / (TP+FP) - "When predict positive, how often correct?"
- **Recall (Sensitivity)** = TP / (TP+FN) - "Of actual positives, how many caught?"
- **Specificity** = TN / (TN+FP) - "Of actual negatives, how many caught?"
- **F1-Score** = 2 × (Precision × Recall) / (Precision + Recall)

**Tradeoffs:**
- High precision → fewer false alarms
- High recall → catch more positives
- Adjust threshold to balance

In [ ]:
# Confusion matrix and metrics
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print('CONFUSION MATRIX:')
print(cm)
print(f'\nBreakdown:')
print(f'True Negatives (TN): {tn}')
print(f'False Positives (FP): {fp}')
print(f'False Negatives (FN): {fn}')
print(f'True Positives (TP): {tp}')

# Calculate metrics
accuracy = (tp + tn) / (tp + tn + fp + fn)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f'\nMETRICS:')
print(f'Accuracy: {accuracy:.3f}')
print(f'Precision: {precision:.3f}')
print(f'Recall (Sensitivity): {recall:.3f}')
print(f'Specificity: {specificity:.3f}')
print(f'F1-Score: {f1:.3f}')

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
           xticklabels=['Predicted No', 'Predicted Yes'],
           yticklabels=['Actual No', 'Actual Yes'])
axes[0].set_title('Confusion Matrix')

# Metrics bar chart
metrics_data = {'Accuracy': accuracy, 'Precision': precision, 
               'Recall': recall, 'F1-Score': f1}
axes[1].bar(metrics_data.keys(), metrics_data.values(), 
           edgecolor='black', alpha=0.7)
axes[1].set_ylabel('Score')
axes[1].set_title('Classification Metrics')
axes[1].set_ylim([0, 1])
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print('\nClassification Report:')
print(classification_report(y_test, y_pred))

## 5.3 ROC Curve and AUC

### Theory

**ROC (Receiver Operating Characteristic) Curve:**
- Plots True Positive Rate (Recall) vs False Positive Rate
- Shows performance across all thresholds
- Diagonal = random classifier

**AUC (Area Under Curve):**
- Summary metric: 0.5 to 1.0
- 0.5 = random, 1.0 = perfect
- >0.8 = good, >0.9 = excellent
- Interpretation: probability model ranks random positive higher than random negative